# Foundational ML: Logistic Regression - Solution

**Problem Statement:** Anova Insurance seeks to classify individuals as **Healthy (0)** or **Unhealthy (1)** using health data, to optimize insurance premium pricing.

**Approach:** Build a Logistic Regression model end-to-end -- from EDA through evaluation and interpretation.

**Dataset:** Healthcare_Dataset_Preprocessed.csv (~9,549 rows, 23 columns)

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, log_loss
)

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 25)
print("Libraries loaded successfully!")

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('Healthcare_Dataset_Preprocessed.csv')

print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
# Missing values check
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values -- dataset is already preprocessed!")

In [ ]:
# Duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
target_counts = df['Target'].value_counts()
print("Target Distribution:")
print(f"  Healthy   (0): {target_counts[0]} ({target_counts[0]/len(df)*100:.1f}%)")
print(f"  Unhealthy (1): {target_counts[1]} ({target_counts[1]/len(df)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['#2ecc71', '#e74c3c']

target_counts.plot(kind='bar', color=colors, ax=axes[0])
axes[0].set_xticklabels(['Healthy (0)', 'Unhealthy (1)'], rotation=0)
axes[0].set_title('Target Count')
axes[0].set_ylabel('Count')

axes[1].pie(target_counts, labels=['Healthy', 'Unhealthy'], colors=colors,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Target Proportion')

plt.tight_layout()
plt.show()

In [ ]:
# Numerical feature distributions by target
num_cols = ['Age', 'BMI', 'Blood_Pressure', 'Cholesterol', 'Glucose_Level',
            'Heart_Rate', 'Sleep_Hours', 'Exercise_Hours', 'Water_Intake', 'Stress_Level']

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for i, col in enumerate(num_cols):
    ax = axes[i // 5, i % 5]
    for label, color in zip([0, 1], ['#2ecc71', '#e74c3c']):
        df[df['Target'] == label][col].hist(bins=30, alpha=0.5, color=color, ax=ax,
                                            label='Healthy' if label == 0 else 'Unhealthy')
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)

fig.suptitle('Numerical Feature Distributions by Health Status', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for numerical features by target
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for i, col in enumerate(num_cols):
    ax = axes[i // 5, i % 5]
    df.boxplot(column=col, by='Target', ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red'))
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')

fig.suptitle('Numerical Features Box Plots (0=Healthy, 1=Unhealthy)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs Target
cat_cols = ['Smoking', 'Alcohol', 'Diet', 'MentalHealth', 'PhysicalActivity',
            'MedicalHistory', 'Allergies']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['Target'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'{col} vs Target', fontsize=11)
    axes[i].set_ylabel('Percentage')
    axes[i].legend(['Healthy', 'Unhealthy'], fontsize=8)
    axes[i].tick_params(axis='x', rotation=0)
axes[-1].set_visible(False)
fig.suptitle('Categorical Features vs Health Status', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(16, 12))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap (Lower Triangle)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with Target
target_corr = corr['Target'].drop('Target').sort_values(key=abs, ascending=False)
print("Feature Correlation with Target (sorted by absolute value):")
print("=" * 45)
for feat, val in target_corr.items():
    bar = '#' * int(abs(val) * 50)
    sign = '+' if val > 0 else '-'
    print(f"  {feat:<22s} {val:+.4f}  {sign} {bar}")

## 4. Data Preprocessing

The dataset is already preprocessed (no missing values, Diet_Type and Blood_Group already one-hot encoded). We still need to:
1. Check for anomalous values
2. Scale features for Logistic Regression

In [ ]:
df_clean = df.copy()

# Check for anomalous values in key columns
print("Checking for anomalous values:")
print(f"  Negative Age values:           {(df_clean['Age'] < 0).sum()}")
print(f"  Negative Exercise_Hours:       {(df_clean['Exercise_Hours'] < 0).sum()}")
print(f"  Age range:                     {df_clean['Age'].min():.1f} to {df_clean['Age'].max():.1f}")
print(f"  Exercise_Hours range:          {df_clean['Exercise_Hours'].min():.1f} to {df_clean['Exercise_Hours'].max():.1f}")

In [ ]:
# Fix negative Exercise_Hours (likely -0.0 from preprocessing)
df_clean['Exercise_Hours'] = df_clean['Exercise_Hours'].clip(lower=0)

print(f"Exercise_Hours range after fix: {df_clean['Exercise_Hours'].min():.1f} to {df_clean['Exercise_Hours'].max():.1f}")
print(f"\nFinal dataset shape: {df_clean.shape}")

## 5. Feature Selection and Train-Test Split

In [ ]:
# Separate features and target
X = df_clean.drop('Target', axis=1)
y = df_clean['Target']

print(f"Features: {X.shape[1]} columns")
print(f"Samples:  {X.shape[0]}")
print(f"\nFeature list: {list(X.columns)}")

In [ ]:
# Train-Test Split (80-20, stratified to maintain class proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTrain target distribution: {dict(y_train.value_counts())}")
print(f"Test target distribution:  {dict(y_test.value_counts())}")

In [ ]:
# Feature Scaling (critical for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("StandardScaler applied.")
print(f"\nSample scaled feature means (should be ~0): {X_train_scaled.mean(axis=0)[:5].round(4)}")
print(f"Sample scaled feature stds  (should be ~1): {X_train_scaled.std(axis=0)[:5].round(4)}")

## 6. Logistic Regression - Model Building

### How Logistic Regression Works

Logistic Regression models the **probability** that an input belongs to a class using the **sigmoid function**:

$$P(y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n)}}$$

- Output is a probability between 0 and 1
- Default threshold: if P >= 0.5 -> predict Unhealthy (1), else Healthy (0)
- Coefficients tell us the direction and strength of each feature's influence

In [ ]:
# Visualize the Sigmoid function
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid, 'b-', linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.7, label='Threshold = 0.5')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('z (linear combination of features)', fontsize=12)
plt.ylabel('P(Unhealthy)', fontsize=12)
plt.title('Sigmoid Function - Heart of Logistic Regression', fontsize=13)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained!")
print(f"Intercept (beta_0): {lr_model.intercept_[0]:.4f}")
print(f"Number of iterations: {lr_model.n_iter_[0]}")

In [ ]:
# Predictions
y_pred = lr_model.predict(X_test_scaled)
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print("Predictions generated.")
print(f"Predicted class distribution: {dict(pd.Series(y_pred).value_counts())}")

## 7. Model Evaluation

In [ ]:
# Core metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
logloss = log_loss(y_test, y_prob)

print("Logistic Regression Performance")
print("=" * 40)
print(f"  Accuracy:   {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  Precision:  {precision:.4f}")
print(f"  Recall:     {recall:.4f}")
print(f"  F1 Score:   {f1:.4f}")
print(f"  ROC AUC:    {roc_auc:.4f}")
print(f"  Log Loss:   {logloss:.4f}")

In [ ]:
# Detailed classification report
print("Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['Healthy (0)', 'Unhealthy (1)']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Healthy', 'Unhealthy'],
            yticklabels=['Healthy', 'Unhealthy'])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Percentages
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=['Healthy', 'Unhealthy'],
            yticklabels=['Healthy', 'Unhealthy'])
axes[1].set_title('Confusion Matrix (% by Row)', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (correctly predicted Healthy):   {tn}")
print(f"False Positives (Healthy predicted Unhealthy):   {fp}")
print(f"False Negatives (Unhealthy predicted Healthy):   {fn}")
print(f"True Positives  (correctly predicted Unhealthy): {tp}")

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'Logistic Regression (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5)')
plt.fill_between(fpr, tpr, alpha=0.1, color='blue')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Logistic Regression', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Predicted probability distribution
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(y_prob[y_test == 0], bins=40, alpha=0.6, color='#2ecc71', label='Actual Healthy (0)')
ax.hist(y_prob[y_test == 1], bins=40, alpha=0.6, color='#e74c3c', label='Actual Unhealthy (1)')
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Decision Threshold (0.5)')
ax.set_xlabel('Predicted Probability of Unhealthy', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Predicted Probabilities', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 8. Model Interpretation - Coefficients Analysis

One of the biggest advantages of Logistic Regression is **interpretability**. Each coefficient tells us:
- **Positive coefficient**: Increasing this feature increases the probability of being Unhealthy
- **Negative coefficient**: Increasing this feature decreases the probability of being Unhealthy
- **Magnitude**: Larger absolute values = stronger influence

In [ ]:
# Coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0],
    'Odds Ratio': np.exp(lr_model.coef_[0])
}).sort_values('Coefficient', ascending=True)

print("Logistic Regression Coefficients & Odds Ratios")
print("=" * 60)
print(f"{'Feature':<24s} {'Coefficient':>12s} {'Odds Ratio':>12s} {'Effect':>10s}")
print("-" * 60)
for _, row in coef_df.iterrows():
    effect = 'Risk +' if row['Coefficient'] > 0 else 'Risk -'
    print(f"  {row['Feature']:<22s} {row['Coefficient']:>+12.4f} {row['Odds Ratio']:>12.4f}   {effect}")

print(f"\n  Intercept: {lr_model.intercept_[0]:+.4f}")

In [ ]:
# Coefficient visualization
plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value', fontsize=12)
plt.title('Logistic Regression Coefficients\n(Red = increases risk, Green = decreases risk)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Top risk factors
print("Top 5 Features that INCREASE Unhealthy risk:")
top_positive = coef_df.sort_values('Coefficient', ascending=False).head(5)
for _, row in top_positive.iterrows():
    print(f"  {row['Feature']:<22s}  coeff: {row['Coefficient']:+.4f}  odds ratio: {row['Odds Ratio']:.4f}")

print(f"\nTop 5 Features that DECREASE Unhealthy risk (protective factors):")
top_negative = coef_df.sort_values('Coefficient', ascending=True).head(5)
for _, row in top_negative.iterrows():
    print(f"  {row['Feature']:<22s}  coeff: {row['Coefficient']:+.4f}  odds ratio: {row['Odds Ratio']:.4f}")

## 9. Cross-Validation

To ensure our model generalizes well and isn't overfitting to the training set.

In [ ]:
# 5-Fold Cross Validation
cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
cv_f1 = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='f1')
cv_auc = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')

print("5-Fold Cross Validation Results")
print("=" * 50)
print(f"  Accuracy:  {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"  F1 Score:  {cv_f1.mean():.4f} +/- {cv_f1.std():.4f}")
print(f"  ROC AUC:   {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

print(f"\n  Per-fold accuracy: {[f'{s:.4f}' for s in cv_scores]}")
print(f"  Test accuracy:     {accuracy:.4f}")
print(f"\n  CV and test scores are close -> model generalizes well." if abs(cv_scores.mean() - accuracy) < 0.02 else "\n  Gap between CV and test may indicate variance issues.")

## 10. Threshold Tuning

For insurance, **missing an unhealthy person is costly** (they might get standard premiums but need expensive care). We may want to optimize the threshold to catch more unhealthy individuals (higher recall).

In [ ]:
# Evaluate different thresholds
thresholds_to_try = np.arange(0.3, 0.75, 0.05)
threshold_results = []

for t in thresholds_to_try:
    y_pred_t = (y_prob >= t).astype(int)
    threshold_results.append({
        'Threshold': t,
        'Accuracy': accuracy_score(y_test, y_pred_t),
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall': recall_score(y_test, y_pred_t, zero_division=0),
        'F1': f1_score(y_test, y_pred_t, zero_division=0),
    })

thresh_df = pd.DataFrame(threshold_results)

plt.figure(figsize=(10, 5))
for col in ['Accuracy', 'Precision', 'Recall', 'F1']:
    plt.plot(thresh_df['Threshold'], thresh_df[col], 'o-', label=col, linewidth=2)
plt.xlabel('Decision Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Metrics vs Decision Threshold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(thresh_df.round(4).to_string(index=False))

In [ ]:
# Optimal threshold based on F1
best_idx = thresh_df['F1'].idxmax()
best_threshold = thresh_df.loc[best_idx, 'Threshold']
print(f"Optimal threshold (best F1): {best_threshold:.2f}")
print(f"  Accuracy:  {thresh_df.loc[best_idx, 'Accuracy']:.4f}")
print(f"  Precision: {thresh_df.loc[best_idx, 'Precision']:.4f}")
print(f"  Recall:    {thresh_df.loc[best_idx, 'Recall']:.4f}")
print(f"  F1:        {thresh_df.loc[best_idx, 'F1']:.4f}")

## 11. Regularization Comparison

Logistic Regression supports different regularization strategies:
- **L1 (Lasso)**: Can zero out coefficients -> feature selection
- **L2 (Ridge)**: Shrinks coefficients -> default, prevents overfitting
- **C parameter**: Inverse regularization strength (smaller C = stronger regularization)

In [ ]:
# Compare L1 vs L2 regularization
reg_results = []

for penalty, solver in [('l1', 'liblinear'), ('l2', 'lbfgs')]:
    for C_val in [0.01, 0.1, 1.0, 10.0]:
        model = LogisticRegression(penalty=penalty, C=C_val, solver=solver,
                                   max_iter=1000, random_state=42)
        model.fit(X_train_scaled, y_train)
        y_p = model.predict(X_test_scaled)
        y_pr = model.predict_proba(X_test_scaled)[:, 1]

        n_nonzero = np.sum(model.coef_[0] != 0)
        reg_results.append({
            'Penalty': penalty.upper(),
            'C': C_val,
            'Accuracy': accuracy_score(y_test, y_p),
            'F1': f1_score(y_test, y_p),
            'ROC AUC': roc_auc_score(y_test, y_pr),
            'Non-zero Features': n_nonzero,
        })

reg_df = pd.DataFrame(reg_results)
print("Regularization Comparison:")
print(reg_df.round(4).to_string(index=False))

## 12. Sample Predictions - Insurance Use Case

In [ ]:
# Demonstrate on sample applicants
n_samples = 10
sample_indices = X_test.sample(n_samples, random_state=42).index

print("Sample Insurance Applicant Predictions")
print("=" * 80)
print(f"{'#':<4s} {'Actual':<12s} {'Predicted':<12s} {'P(Unhealthy)':<14s} {'Premium Rec.':<15s} {'Correct?'}")
print("-" * 80)

for i, idx in enumerate(sample_indices, 1):
    actual = y_test.loc[idx]
    sample_scaled = scaler.transform(X_test.loc[[idx]])
    pred = lr_model.predict(sample_scaled)[0]
    prob = lr_model.predict_proba(sample_scaled)[0][1]

    actual_str = 'Unhealthy' if actual == 1 else 'Healthy'
    pred_str = 'Unhealthy' if pred == 1 else 'Healthy'
    correct = 'Yes' if actual == pred else 'NO'

    if prob < 0.3:
        premium = 'Standard'
    elif prob < 0.6:
        premium = 'Moderate'
    else:
        premium = 'High'

    print(f"{i:<4d} {actual_str:<12s} {pred_str:<12s} {prob:<14.4f} {premium:<15s} {correct}")

## 13. Conclusion

### Summary

| Step | Details |
|------|----------|
| **EDA** | Explored distributions, class balance, feature-target relationships, correlations |
| **Preprocessing** | Data was pre-cleaned; fixed negative Exercise_Hours, applied StandardScaler |
| **Model** | Logistic Regression with L2 regularization (default) |
| **Evaluation** | Accuracy, Precision, Recall, F1, ROC AUC, Confusion Matrix |
| **Interpretation** | Analyzed coefficients and odds ratios to identify key risk factors |
| **Cross-Validation** | 5-fold CV to verify generalization |
| **Threshold Tuning** | Explored different decision thresholds for insurance context |
| **Regularization** | Compared L1 vs L2 with different C values |

### Why Logistic Regression for This Problem?

1. **Interpretable**: Coefficients directly tell us which features increase/decrease health risk
2. **Probabilistic**: Outputs probability scores, not just classes -- useful for tiered premium pricing
3. **Fast & Scalable**: Trains quickly even on large datasets
4. **Regulatory Friendly**: Insurance decisions often require explainable models

### Business Recommendation for Anova Insurance

- Use the predicted probability to create **tiered premium pricing**:
  - P(Unhealthy) < 0.3 -> Standard premium
  - P(Unhealthy) 0.3 - 0.6 -> Moderate premium
  - P(Unhealthy) > 0.6 -> High premium
- Key risk factors identified by the model can guide **health screening priorities**
- Consider lowering the decision threshold to reduce false negatives (missing unhealthy individuals)